<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/07_worked_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 · Worked analysis — a complete microbiome study, start to finish

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: R* - Instructor: Anderson Santos

## Setup — run this first

This cell installs the R packages the lessons use and makes the example data available. On **Google Colab** it just works: run it (Shift+Enter) and wait until it prints **Ready**. You do not need to understand it.

In [ ]:
# Setup — run this first. Works on Google Colab and on a local Jupyter with R.
# It installs the packages the lessons use and makes the example data available.

pkgs <- c("readr", "dplyr", "tidyr", "tibble", "ggplot2", "forcats", "stringr", "vegan")
to_install <- setdiff(pkgs, rownames(installed.packages()))
if (length(to_install) > 0) {
  # On Colab (Linux) use Posit Public Package Manager binaries: ~1-2 min instead of
  # compiling from source (~15-20 min).
  if (grepl("linux", R.version$os) && file.exists("/etc/os-release")) {
    cn <- grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE)
    cn <- gsub('VERSION_CODENAME=|"', "", cn)
    if (length(cn) == 1 && nzchar(cn)) {
      options(repos = c(CRAN = sprintf("https://p3m.dev/cran/__linux__/%s/latest", cn)))
      options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
              paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
    }
  }
  install.packages(to_install)
}

# Fetch the example data if it is not already next to the notebook (the Colab case).
if (!file.exists("../data/sample_metadata.csv") && !file.exists("data/sample_metadata.csv")) {
  system("git clone -q https://github.com/andersonavilasantos/ufz-summerschool-r.git course_repo")
  setwd("course_repo/notebooks")
}

# Load the packages quietly and keep read_csv output tidy (cleaner notebook output).
suppressPackageStartupMessages(invisible(lapply(pkgs, require, character.only = TRUE)))
options(readr.show_col_types = FALSE)

cat("Ready. Data folder:", if (file.exists("../data/sample_metadata.csv")) "../data" else "data", "\n")

In [ ]:
#github commit

## What this notebook is

This is a fully worked, read-and-run analysis, the reference companion to the
fill-in **05 · Capstone**. Every cell is already written: read it top to bottom
and run it to follow a complete microbial-ecology study on the real HITChip Atlas
human-gut dataset. Nothing is hidden behind a blank.

> **Instructor note.** Use this two ways: (1) as the instructor's live demo, where
> you run it once and then walk the class through each figure; or (2) as a worked
> solution students read after attempting the capstone in pairs. The capstone (05)
> asks you to write this; here you get to study it.

The arc is the same set of steps you will repeat all week: import, clean,
wrangle, summarise, visualise, test, ordinate, export.

Everything uses only tools from lessons **01–04** (readr, dplyr, tidyr, ggplot2,
forcats) plus **vegan** for diversity and ordination. CRAN only.

---

## 1 · Import — load the three tables

The study ships as three tidy tables: participant **metadata**, a **taxonomy**
(genus → phylum/family), and a wide **abundance** matrix (1,006 samples × 130
genera).

In [ ]:
library(readr); library(dplyr); library(tidyr)      # import, wrangle, reshape
library(ggplot2); library(forcats); library(stringr)  # plot, reorder factors, string helpers
library(vegan)     # community ecology: diversity + ordination

# Works whether the notebook runs from notebooks/ or from the project root.
data_dir <- if (file.exists("../data/sample_metadata.csv")) "../data" else "data"
meta     <- read_csv(file.path(data_dir, "sample_metadata.csv"))  # one row per participant
taxonomy <- read_csv(file.path(data_dir, "taxonomy.csv"))         # genus -> phylum / family
abund    <- read_csv(file.path(data_dir, "genus_abundance.csv"))  # wide: sample_id + one col per genus

# cat() prints the dimensions so we can sanity-check each table loaded correctly.
cat("metadata :", nrow(meta),  "x", ncol(meta),  "\n")
cat("taxonomy :", nrow(taxonomy), "x", ncol(taxonomy), "\n")
cat("abundance:", nrow(abund), "x", ncol(abund), "\n")

A quick look at what we are working with:

In [ ]:
glimpse(meta)   # glimpse() = a transposed preview: every column, its type, and the first values

---

## 2 · Clean — deal with the missing values

Real data is never complete. Before analysing, we (a) measure the gaps and (b)
build an analysis-ready copy that keeps only samples with a known `age` **and**
`bmi_group`, and turns `bmi_group` into an **ordered factor** so tables and plots
respect the natural underweight → morbidly-obese order.

In [ ]:
# How much is missing?
cat("missing age     :", sum(is.na(meta$age)),       "\n")
cat("missing bmi_group:", sum(is.na(meta$bmi_group)), "\n")
cat("missing sex     :", sum(is.na(meta$sex)),        "\n")

# The natural order of BMI categories (a plain vector we hand to factor() below).
bmi_levels <- c("underweight","lean","overweight",
                "obese","severeobese","morbidobese")

meta_clean <- meta |>
  filter(!is.na(age), !is.na(bmi_group)) |>                    # keep only fully-labelled samples
  # factor(..., levels=) makes bmi_group ordered: plots/tables follow underweight -> morbidobese,
  # not alphabetical order (a classic R gotcha — character columns sort A-Z by default).
  mutate(bmi_group = factor(bmi_group, levels = bmi_levels))

cat("kept", nrow(meta_clean), "of", nrow(meta), "samples after cleaning\n")

> **Instructor note.** Cleaning is not optional busywork. Dropping the unlabelled
> rows now means every downstream `group_by` is honest. Show the class the
> before/after sample count so they see the cost of missing data.

---

## 3 · Wrangle — reshape and join into one tidy table

The abundance table is **wide** (one column per genus). Most tidyverse verbs want
**long** data, so we `pivot_longer`, then `left_join` the taxonomy (to get each
genus's phylum) and `inner_join` the cleaned metadata (dropping samples we cleaned
away).

In [ ]:
ab_long <- abund |>
  pivot_longer(-sample_id, names_to = "genus", values_to = "abundance")

cat(nrow(ab_long), "rows =", nrow(abund), "samples x",
    ncol(abund) - 1, "genera\n")

ab_full <- ab_long |>
  left_join(taxonomy, by = "genus") |>                         # + phylum / family
  inner_join(select(meta_clean, sample_id, nationality, bmi_group, age),
             by = "sample_id")                                 # keep cleaned samples

glimpse(ab_full)

---

## 4 · Summarise — alpha diversity per sample

**Alpha diversity** describes how varied a single community is. For every sample
we compute **observed richness** (how many genera are present) and the **Shannon
index** (richness weighted by evenness), then look at the mean per region.

In [ ]:
# One row per sample. group_by keeps the metadata columns alongside each sample.
alpha <- ab_full |>
  group_by(sample_id, nationality, bmi_group, age) |>
  summarise(
    richness = sum(abundance > 0),                       # count of genera with count > 0 (TRUE sums as 1)
    shannon  = diversity(abundance, index = "shannon"),  # vegan::diversity() = Shannon index
    .groups  = "drop")                                   # drop grouping so `alpha` is a flat table

# Now summarise those per-sample numbers up to a mean per region, most-diverse first.
alpha |>
  group_by(nationality) |>
  summarise(mean_richness = round(mean(richness), 1),
            mean_shannon  = round(mean(shannon), 2),
            n = n()) |>                        # n() = how many samples in each region
  arrange(desc(mean_shannon))                  # sort highest diversity first

---

## 5 · Visualise (I) — diversity across BMI groups

The first figure: does gut diversity change across body-mass categories? A
**boxplot** shows the distribution per group; faint jittered points keep us honest
about how many samples sit behind each box.

In [ ]:
p_div <- ggplot(alpha, aes(bmi_group, shannon, fill = bmi_group)) +
  geom_boxplot(outlier.alpha = 0.3) +
  geom_jitter(width = 0.15, alpha = 0.10, size = 0.4) +
  labs(title = "Gut diversity across BMI groups",
       subtitle = "HITChip Atlas — 1,006 healthy adults",
       x = "BMI group", y = "Shannon diversity") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 20, hjust = 1))
p_div

---

## 6 · Visualise (II) — taxonomic composition per region

The classic microbiome figure. We compute, per sample, each **phylum's** share of
the community, average those shares within each region, and draw a **stacked bar**
(y as %). The Firmicutes/Bacteroidetes balance is the headline.

In [ ]:
composition <- ab_full |>
  group_by(sample_id, nationality, phylum) |>
  summarise(phy = sum(abundance), .groups = "drop") |>  # total reads per phylum in each sample
  group_by(sample_id) |>
  mutate(rel = phy / sum(phy)) |>                     # within-sample proportion (normalises depth)
  group_by(nationality, phylum) |>
  summarise(mean_rel = mean(rel), .groups = "drop")   # average that proportion across each region

p_comp <- ggplot(composition,
                 aes(nationality, mean_rel,
                     fill = fct_reorder(phylum, mean_rel, sum))) +
  geom_col() +
  scale_y_continuous(labels = scales::percent) +
  labs(title = "Phylum composition across regions",
       x = NULL, y = "Mean relative abundance", fill = "Phylum") +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))
p_comp

# The single dominant phylum in each region:
composition |> group_by(nationality) |> slice_max(mean_rel, n = 1)

> **Instructor note.** Point out that Firmicutes dominates everywhere, exactly
> what the literature reports for the healthy Western gut, and that the
> Firmicutes/Bacteroidetes ratio is the variable most microbiome papers focus on.

---

## 7 · Test — is the diversity difference real?

Eyeballing a boxplot is not a statistic. A **one-way ANOVA** asks whether mean
Shannon diversity differs across BMI groups more than we would expect by chance.

In [ ]:
# aov() fits the model "shannon explained by bmi_group"; the ~ reads as "modelled by".
fit <- aov(shannon ~ bmi_group, data = alpha)
summary(fit)   # the ANOVA table: look at the F value and its Pr(>F) p-value

Read the **`Pr(>F)`** column: a small p-value (say < 0.05) is evidence that at
least one BMI group differs in mean diversity. Always report the statistic
**together** with the Step 5 boxplot — the picture shows the effect size, the test
shows whether it is distinguishable from noise.

**A second angle — *which* groups differ?** ANOVA only says "at least one group
differs". **Tukey's HSD** does the follow-up pairwise comparisons on the *same*
fitted model, with the p-values adjusted for multiple testing.

In [ ]:
# TukeyHSD() takes the fitted aov object and compares every pair of BMI groups.
tuk <- TukeyHSD(fit)

# Keep the comparisons and show the most significant ones (smallest adjusted p first).
as.data.frame(tuk$bmi_group) |>
  tibble::rownames_to_column("comparison") |>   # the "groupA-groupB" labels become a column
  arrange(`p adj`) |>                            # `p adj` = p-value adjusted for many comparisons
  head(5)

---

## 8 · Ordinate — NMDS of whole communities

Alpha diversity is one number per sample; **beta diversity** compares *whole
communities*. **NMDS** (non-metric multidimensional scaling) on **Bray–Curtis**
distances places each sample in 2-D so that similar communities sit close
together. We colour by region to see whether geography structures the gut
microbiome.

In [ ]:
# A balanced subset keeps the ordination fast enough for a live class.
set.seed(1)
sub_ids <- meta_clean |>
  filter(nationality %in% c("Scandinavia","US","SouthEurope","CentralEurope")) |>
  group_by(nationality) |>
  slice_sample(n = 40) |>
  pull(sample_id)

mat <- abund |>
  filter(sample_id %in% sub_ids) |>
  tibble::column_to_rownames("sample_id") |>
  as.matrix()
mat <- mat[rowSums(mat) > 0, ]                 # drop any all-zero samples

nmds <- metaMDS(mat, distance = "bray", trace = FALSE, try = 20)

In [ ]:
scores_df <- as.data.frame(scores(nmds, display = "sites"))
scores_df$sample_id <- rownames(scores_df)
scores_df <- left_join(scores_df,
                       select(meta_clean, sample_id, nationality),
                       by = "sample_id")

p_nmds <- ggplot(scores_df, aes(NMDS1, NMDS2, colour = nationality)) +
  geom_point(size = 2, alpha = 0.85) +
  stat_ellipse(level = 0.7) +                  # 70% confidence ellipse per region
  labs(title = paste0("NMDS of gut communities (stress = ",
                      round(nmds$stress, 3), ")"),
       colour = "Region") +
  theme_minimal(base_size = 12)
p_nmds

> **Instructor note.** A stress below about 0.2 means the 2-D picture is a fair
> summary of the true distances. Overlapping ellipses mean communities are similar
> across regions; separated ellipses mean regional structure. This is the same
> ordination logic as the PCA in lesson 06, with a different distance.

---

## 9 · Export — save the figures and the result tables

A real analysis leaves **artefacts** a collaborator can reuse: the figures and the
summary tables. `ggsave()` writes plots; `write_csv()` writes tables.

In [ ]:
# Write figures into the project figures/ folder (created if missing).
fig_dir <- if (dir.exists("../figures")) "../figures" else "figures"
dir.create(fig_dir, showWarnings = FALSE)

ggsave(file.path(fig_dir, "worked_diversity_by_bmi.png"),
       p_div,  width = 7, height = 4.5, dpi = 150)
ggsave(file.path(fig_dir, "worked_phylum_composition.png"),
       p_comp, width = 8, height = 5,   dpi = 150)
ggsave(file.path(fig_dir, "worked_nmds_regions.png"),
       p_nmds, width = 7, height = 5,   dpi = 150)

# Write the two result tables next to this notebook.
write_csv(alpha,       "worked_alpha_diversity.csv")
write_csv(composition, "worked_phylum_by_region.csv")

cat("Saved 3 figures to", normalizePath(fig_dir), "\n")
cat("Wrote worked_alpha_diversity.csv and worked_phylum_by_region.csv\n")

---

## Recap — the whole analysis

We took a real, published human-gut dataset from raw tables to defensible
conclusions:

- **Imported** three tables and **cleaned** away the unlabelled samples.
- **Wrangled** a wide matrix into tidy long form and **joined** taxonomy + metadata.
- **Summarised** alpha diversity, then **visualised** it (boxplot) and the
  **phylum composition** per region (stacked bar).
- **Tested** the diversity difference with an ANOVA and **ordinated** whole
  communities with **NMDS**.
- **Exported** three figures and two result tables.

Every verb came from lessons 01–04; the only new idea was chaining them into a
study. Now flip to **05 · Capstone** and write this yourself, or swap in your own
feature table this week, and most of this code still applies.